In [1]:
!pip install sdv xgboost imbalanced-learn umap-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.7/204.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/thesis_datasets/'
LOG_PATH  = '/content/drive/MyDrive/thesis_logs_tvae_gc/'

import os
os.makedirs(LOG_PATH, exist_ok=True)
print('Drive mounted')

Mounted at /content/drive
Drive mounted


In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, PredefinedSplit
from xgboost import XGBClassifier
from tqdm.notebook import tqdm
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer, GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata
from sdv.sampling import Condition
import time

print('All imports OK')

All imports OK


In [4]:
target   = 'target'
cat_cols = []
seeds    = [220, 320, 420]

param_grids_lr  = [{'solver': ['liblinear', 'lbfgs'], 'max_iter': [200, 500, 1500]}]
param_grids_rf  = [{'n_estimators': [150, 250], 'max_depth': [15, 25, None]}]
param_grids_dt  = [{'max_depth': [15, 25, None]}]
param_grids_mlp = [{'learning_rate': ['constant', 'adaptive'],
                    'hidden_layer_sizes': [(100,), (100, 100), (200,), (150, 100)],
                    'max_iter': [200, 500, 1500]}]
param_grids_xgb = [{'max_depth': [6, 10, 15], 'n_estimators': [100, 150, 250]}]
param_grids_gs  = [{'var_smoothing': [1e-9]}]

In [5]:
def preprocess_default(filepath):
    return pd.read_csv(filepath).dropna().reset_index(drop=True)

def preprocess_hcv(filepath):
    df = pd.read_csv(filepath)
    df['Sex'] = df['Sex'].map({'m': 1, 'f': 0})
    return df.dropna().reset_index(drop=True)

def preprocess_ilpd(filepath):
    df = pd.read_csv(filepath)
    df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
    return df.dropna().reset_index(drop=True)

datasets_config = {
    #'bcwd_10': preprocess_default(DATA_PATH + 'bcwd_10.csv'),
    #'bcwd_25': preprocess_default(DATA_PATH + 'bcwd_25.csv'),
    #'bcwd_50': preprocess_default(DATA_PATH + 'bcwd_50.csv'),
    #'bcwp_10': preprocess_default(DATA_PATH + 'bcwp_10.csv'),
    'bcwp_25': preprocess_default(DATA_PATH + 'bcwp_25.csv'),
    'bcwp_50': preprocess_default(DATA_PATH + 'bcwp_50.csv'),
    'bcc_10':  preprocess_default(DATA_PATH + 'bcc_10.csv'),
    'bcc_25':  preprocess_default(DATA_PATH + 'bcc_25.csv'),
    'bcc_50':  preprocess_default(DATA_PATH + 'bcc_50.csv'),
    #'arcene_10': preprocess_default(DATA_PATH + 'arcene_10.csv'),
    #'arcene_15': preprocess_default(DATA_PATH + 'arcene_15.csv'),
    #'arcene_50': preprocess_default(DATA_PATH + 'arcene_50.csv'),
    'pima_10': preprocess_default(DATA_PATH + 'pima_10.csv'),
    'pima_25': preprocess_default(DATA_PATH + 'pima_25.csv'),
    'pima_50': preprocess_default(DATA_PATH + 'pima_50.csv'),
    'hcv_10':  preprocess_hcv(DATA_PATH + 'hcv_10.csv'),
    'hcv_25':  preprocess_hcv(DATA_PATH + 'hcv_25.csv'),
    'hcv_50':  preprocess_hcv(DATA_PATH + 'hcv_50.csv'),
    'ilpd_5':  preprocess_ilpd(DATA_PATH + 'ilpd_5.csv'),
    'ilpd_10': preprocess_ilpd(DATA_PATH + 'ilpd_10.csv'),
    'ilpd_15': preprocess_ilpd(DATA_PATH + 'ilpd_15.csv'),
    'parkinsons_5':  preprocess_default(DATA_PATH + 'parkinsons_5.csv'),
    'parkinsons_10': preprocess_default(DATA_PATH + 'parkinsons_10.csv'),
    'parkinsons_15': preprocess_default(DATA_PATH + 'parkinsons_15.csv'),
}

print(f'Loaded {len(datasets_config)} datasets')

Loaded 19 datasets


In [6]:
def get_sdv_metadata(data, target, cat_cols):
    metadata = SingleTableMetadata()
    for col in data.columns:
        if col == target or col in cat_cols:
            metadata.add_column(col, sdtype='categorical')
        else:
            metadata.add_column(col, sdtype='numerical')
    return metadata

def generate_sdv_model(data, target, cat_cols=[], model='ctgan', seed=22):
    data = data.copy().reset_index(drop=True)
    data.columns = data.columns.astype(str)
    data[target] = data[target].astype(str)
    for col in cat_cols:
        data[col] = data[col].astype(str)

    metadata = get_sdv_metadata(data, target, cat_cols)

    if model == 'ctgan':
        synthesizer = CTGANSynthesizer(metadata, epochs=300, cuda=True)
    elif model == 'tvae':
        synthesizer = TVAESynthesizer(metadata, epochs=300, cuda=True)
    elif model == 'copula':
        synthesizer = GaussianCopulaSynthesizer(metadata)

    synthesizer.fit(data)

    final_data_gen = []
    for cls in data[target].unique():
        n_rows = data[data[target] == cls].shape[0]
        condition = Condition(
            num_rows=n_rows,
            column_values={target: cls}
        )
        synthetic = synthesizer.sample_from_conditions(conditions=[condition])
        synthetic[target] = cls
        final_data_gen.append(synthetic)

    final_data_gen = pd.concat(final_data_gen, ignore_index=True)
    final_data_gen.columns = final_data_gen.columns.astype(str)
    final_data_gen[target] = final_data_gen[target].astype(int)
    data[target] = data[target].astype(int)

    return final_data_gen.reset_index(drop=True)

In [7]:
for dataset_name, data_raw in tqdm(datasets_config.items(), desc='Datasets'):

    log_file = LOG_PATH + f'{dataset_name}_log_sdv.csv'

    if os.path.exists(log_file):
        print(f'Skipping {dataset_name} — already done')
        continue

    data = data_raw.copy().reset_index(drop=True)
    data.columns = data.columns.astype(str)
    num_cols = [col for col in data.columns if col != target]
    storage = pd.DataFrame()

    for seed in tqdm(seeds, desc='Seeds', leave=False):
        counter = 0
        skf = StratifiedKFold(n_splits=5, random_state=seed, shuffle=True)

        for train_index, test_index in tqdm(skf.split(
            data[[col for col in data.columns if col != target]],
            data[target]
        ), leave=False, desc='Folds'):

            time1 = time.time()
            train = data.loc[train_index, :].copy().reset_index(drop=True)
            test  = data.loc[test_index,  :].copy().reset_index(drop=True)

            minority_count = train[train[target] == 1].shape[0]
            if minority_count < 2:
                print(f'Skipping {dataset_name} fold {counter} — too few minority instances')
                counter += 1
                continue

            try:
              data_copula = generate_sdv_model(train.copy(), target, cat_cols.copy(), model='copula', seed=seed)
              data_copula = data_copula.dropna().reset_index(drop=True)
            except Exception as e:
              print(f'COPULA FAILED {dataset_name} fold {counter}: {e}')
              data_copula = pd.DataFrame(columns=train.columns)

            try:
                data_ctgan = generate_sdv_model(train.copy(), target, cat_cols.copy(), model='ctgan', seed=seed)
                data_ctgan = data_ctgan.dropna().reset_index(drop=True)
            except Exception as e:
                print(f'CTGAN FAILED {dataset_name} fold {counter}: {e}')
                data_ctgan = pd.DataFrame(columns=train.columns)

            try:
                data_tvae = generate_sdv_model(train.copy(), target, cat_cols.copy(), model='tvae', seed=seed)
                data_tvae = data_tvae.dropna().reset_index(drop=True)
            except Exception as e:
                print(f'TVAE FAILED {dataset_name} fold {counter}: {e}')
                data_tvae = pd.DataFrame(columns=train.columns)

            for dataset, title in zip(
                [data_ctgan, data_tvae, data_copula, train],
                ['ctgan',    'tvae',    'copula',    'train']
            ):
                if dataset.shape[0] == 0:
                    print(f'Skipping {title} — empty dataset')
                    continue

                temp = pd.DataFrame()
                available_num_cols = [col for col in num_cols if col in dataset.columns]

                bin_test_adapted = test.copy()
                bin_test_adapted[[col for col in dataset.columns
                                   if col not in bin_test_adapted.columns]] = 0
                bin_test_adapted = bin_test_adapted[dataset.columns]

                scaler = MinMaxScaler()
                scaler.fit(dataset[available_num_cols])

                X_train_scaled = pd.DataFrame(
                    scaler.transform(dataset[available_num_cols]),
                    index=dataset.index, columns=available_num_cols
                )
                X_test_scaled = pd.DataFrame(
                    scaler.transform(bin_test_adapted[available_num_cols]),
                    index=bin_test_adapted.index, columns=available_num_cols
                )

                X_train_scaled[target] = dataset[target].values
                X_test_scaled[target]  = bin_test_adapted[target].values

                X = pd.concat([X_train_scaled, X_test_scaled]).reset_index(drop=True)
                y = X[target]
                X = X[[col for col in X.columns if col != target]]

                if X.isnull().any().any():
                    print(f'NaN found for {dataset_name} {title} fold {counter} — skipping')
                    continue

                separation = np.where(
                    X.index <= X_train_scaled.reset_index(drop=True).index.max(), -1, 0
                )
                ps = PredefinedSplit(separation)

                model1 = LogisticRegression(random_state=seed, n_jobs=1)
                model2 = RandomForestClassifier(n_jobs=1, random_state=seed)
                model3 = DecisionTreeClassifier(random_state=seed)
                model4 = MLPClassifier(random_state=seed)
                model5 = XGBClassifier(seed=seed)
                model6 = GaussianNB()

                for model, par, name_model in zip(
                    [model1, model2, model3, model4, model5, model6],
                    [param_grids_lr, param_grids_rf, param_grids_dt,
                     param_grids_mlp, param_grids_xgb, param_grids_gs],
                    ['lr', 'rf', 'dt', 'mlp', 'xgb', 'gs']
                ):
                    model_search_cv = GridSearchCV(
                        model, par, cv=ps, scoring='f1', n_jobs=1, refit=False
                    )
                    model_search_cv.fit(X, y)
                    temp_r = pd.DataFrame(model_search_cv.cv_results_)
                    temp_r['Model'] = name_model
                    temp = pd.concat([temp, temp_r], ignore_index=True)

                scaler_priv = MinMaxScaler()
                scaler_priv.fit(train[available_num_cols])
                original = scaler_priv.transform(train[available_num_cols])
                searcher = scaler_priv.transform(test[available_num_cols]) if title == 'train' else scaler_priv.transform(dataset[available_num_cols])

                nbrs = NearestNeighbors(n_neighbors=10, n_jobs=1).fit(original)
                distances, _ = nbrs.kneighbors(searcher)

                temp['Mean_Knn']         = np.nanmean(distances[:, 0])
                temp['10_Knn']           = np.nanquantile(distances[:, 0], 0.1)
                temp['Mean_Ratio_Knn']   = np.nanmean(distances[:, 0] / distances[:, 1])
                temp['Mean_Ratio_10Knn'] = np.nanmean(distances[:, 0] / distances[:, 9])
                temp['10_Ratio_Knn']     = np.nanquantile(distances[:, 0] / distances[:, 1], 0.1)
                temp['10_Ratio_10Knn']   = np.nanquantile(distances[:, 0] / distances[:, 9], 0.1)
                temp['Duplicates']       = (distances[:, 0] == 0).sum()
                temp['Generator']        = title
                temp['Seed']             = seed
                temp['fold']             = counter
                temp['dataset']          = dataset_name

                storage = pd.concat([storage, temp], ignore_index=True)

            time2 = time.time()
            print(f'{dataset_name} | Fold {counter} done in {time2-time1:.1f}s')
            counter += 1

        # Save after every seed
        storage.to_csv(log_file, index=False)
        print(f'Checkpoint saved: {log_file}')

    print(f'Done: {dataset_name}')

Datasets:   0%|          | 0/19 [00:00<?, ?it/s]

Seeds:   0%|          | 0/3 [00:00<?, ?it/s]

Folds: 0it [00:00, ?it/s]




  0%|          | 0/285 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/285 [00:00<?, ?it/s]


Sampling conditions: 100%|██████████| 285/285 [00:00<00:00, 1426.85it/s]



  0%|          | 0/85 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/85 [00:00<?, ?it/s]


Sampling conditions: 100%|██████████| 85/85 [00:00<00:00, 504.19it/s]



  0%|          | 0/285 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/285 [00:00<?, ?it/s]


Sampling conditions:  56%|█████▌    | 160/285 [00:00<00:00, 629.95it/s]


Sampling conditions: 100%|██████████| 285/285 [00:00<00:00, 563.46it/s]



  0%|          | 0/85 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/85 [00:00<?, ?it/s]


Sampling conditions:  36%|███▋      | 31/85 [00:00<00:00, 119.15it/s]


Sampling conditions: 100%|██████████| 85/85 [00:00<00:00, 175.54it/s]



  0%|          | 0/285 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/285 [00:00<?, ?it/s]


Sampling conditions: 100%|███████

bcwd_50 | Fold 0 done in 217.1s


...

TVAE FAILED bcwp_25 fold 2: Unable to sample any rows for the given conditions. Try increasing `max_tries_per_batch` (currently: 100). Note that increasing this value will also increase the sampling time.
Skipping tvae — empty dataset
bcwp_25 | Fold 2 done in 95.0s





  0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions: 100%|██████████| 119/119 [00:00<00:00, 412.90it/s]



  0%|          | 0/8 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/8 [00:00<?, ?it/s]


Sampling conditions: 100%|██████████| 8/8 [00:00<00:00, 27.40it/s]



  0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions:  71%|███████▏  | 85/119 [00:00<00:00, 335.47it/s]


Sampling conditions: 100%|██████████| 119/119 [00:00<00:00, 166.90it/s]



  0%|          | 0/8 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/8 [00:00<?, ?it/s]


Sampling conditions:  62%|██████▎   | 5/8 [00:00<00:00, 17.39it/s]


Sampling conditions:  88%|████████▊ | 7/8 [00:00<00:00,  8.55it/s]


Sampling conditions: 100%|██████████| 8/8 [00:00<00:00,  8.27it/s]



  0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/11

TVAE FAILED bcwp_25 fold 3: Unable to sample any rows for the given conditions. Try increasing `max_tries_per_batch` (currently: 100). Note that increasing this value will also increase the sampling time.
Skipping tvae — empty dataset
bcwp_25 | Fold 3 done in 92.3s





  0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/119 [00:00<?, ?it/s]


Sampling conditions: 100%|██████████| 119/119 [00:00<00:00, 561.85it/s]



  0%|          | 0/9 [00:00<?, ?it/s]


Sampling conditions:   0%|          | 0/9 [00:00<?, ?it/s]


Sampling conditions: 100%|██████████| 9/9 [00:00<00:00, 49.06it/s]


KeyboardInterrupt: 